# 05 - PyCaret Workflow vs Manual Workflow

## Why PyCaret exists
PyCaret accelerates experimentation with a compact API and built-in model comparison.

## Tradeoff
- Faster experimentation and less boilerplate
- Lower transparency and lower control than explicit sklearn pipelines


## Definition
PyCaret accelerates model comparison while manual pipelines retain full control and transparency.

## Theory
This section explains the statistical or ML theory behind the technique and why it is useful in credit recovery operations.

## Mathematical Intuition
We translate the idea into score/probability/ranking logic so each metric can be interpreted quantitatively.

## Financial Intuition
We connect the method to borrower affordability, delinquency severity, collateral protection, and expected recoverable cashflows.

## Business Impact
We explain what this enables for collection managers, risk teams, and executives.

## Real-World Example
Teams often prototype in PyCaret, then harden the selected model family in explicit sklearn code.

## Visual Explanation
Charts in this notebook show how model/segment behavior changes across borrower groups.

## Code Explanation
Every code cell below is paired with interpretation so beginners can connect implementation details to business outcomes.

## Interpretation of Results
We summarize what changed, why it matters, and how to act on it.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.loan_recovery import (
    DATA_PATH,
    FIGURES_DIR,
    MODELS_DIR,
    TABLES_DIR,
    REPORTS_DIR,
    LoanDataLoader,
    FeatureEngineer,
    LoanEDA,
    BorrowerSegmenter,
    ModelTrainer,
    ModelEvaluator,
    RecoveryStrategyEngine,
    ModelExplainer,
    DashboardBuilder,
    LazyPredictBenchmark,
    PyCaretWorkflow,
    FLAMLOptimizer,
    SmartLoanRecoveryPipeline,
    load_model,
)

sns.set_theme(style="whitegrid")


In [ ]:
def ensure_pipeline_artifacts() -> None:
    required = [
        TABLES_DIR / "manual_model_leaderboard.csv",
        TABLES_DIR / "feature_enriched_data.csv",
        MODELS_DIR / "best_manual_model.joblib",
    ]
    if not all(path.exists() for path in required):
        pipeline = SmartLoanRecoveryPipeline(data_path=DATA_PATH, random_state=42)
        pipeline.run()

ensure_pipeline_artifacts()


In [ ]:
enriched = pd.read_csv(TABLES_DIR / "feature_enriched_data.csv")
manual = pd.read_csv(TABLES_DIR / "manual_model_leaderboard.csv")

pycaret = PyCaretWorkflow(random_state=42)
py_out = pycaret.run(enriched.drop(columns=["Borrower_ID"]), target_col="Recovery_Status")

display(py_out.comparison_table)
display(manual)


In [ ]:
manual_best = manual.iloc[0]
pycaret_best = py_out.comparison_table.iloc[0]
comparison = pd.DataFrame(
    [
        {
            "workflow": "Manual",
            "model": manual_best["model"],
            "accuracy": manual_best["accuracy"],
            "f1": manual_best["f1_weighted"],
            "roc_auc": manual_best["roc_auc_ovr"],
        },
        {
            "workflow": "PyCaret",
            "model": pycaret_best["Model"],
            "accuracy": pycaret_best["Accuracy"],
            "f1": pycaret_best["F1"],
            "roc_auc": pycaret_best["AUC"],
        },
    ]
)
display(comparison)


## Interpretation
PyCaret should be treated as a strong accelerator for discovery, then hardened with explicit pipelines for production.
